# CFLOW Two-Stage on Severstal (Protocol-Aligned)

This notebook keeps a **two-stage** setup: Stage-1 CFLOW anomaly screening + Stage-2 ResNet known-class classifier.

It matches the one-stage operating-point protocol with FPR targets `[0.05, 0.10, 0.15, 0.20]` and uses **validation-normal only** for stage-1 threshold calibration.

In [ ]:
# 0) Runtime preflight (hard fail on incompatible stack)
import os, sys
import numpy as np
import torch, torchvision

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('Torch:', torch.__version__)
print('TorchVision:', torchvision.__version__)

msg = (
    'Incompatible runtime. Required stack: numpy==1.26.x, '
    'torch==2.2.2(+cu121), torchvision==0.17.2(+cu121). '
    'Install pinned versions and restart runtime before continuing.'
)
assert np.__version__.startswith('1.26'), msg
assert torch.__version__.startswith('2.2.2'), msg
assert torchvision.__version__.startswith('0.17.2'), msg
print('Preflight OK')


In [ ]:
# 1) Global config
from pathlib import Path

PILOT_MODE = True  # set False for heavier full run
SEED = 42
FPR_GRID = [0.05, 0.10, 0.15, 0.20]
assert 0.15 in FPR_GRID

DRIVE_ROOT = Path('/content/drive/MyDrive')
DATA_ROOT = DRIVE_ROOT / 'datasets' / 'severstal'
OUT_ROOT = DRIVE_ROOT / 'fyp_outputs' / 'severstral_cflow_two_stage_full'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

FYP_REPO = Path('/content/FYP-code')
CFLOW_REPO = Path('/content/cflow-ad')

# runtime class must be an official MVTec class in cflow-ad
CFLOW_RUNTIME_CLASS = 'bottle'

# split labels from severstal-osr config
SPLITS = ['a', 'b', 'c', 'd']

# stage-2 unknown threshold (known-val confidence lower-tail)
STAGE2_UNKNOWN_Q = 0.10

print('PILOT_MODE:', PILOT_MODE)
print('FPR grid:', FPR_GRID)
print('OUT_ROOT:', OUT_ROOT)


In [ ]:
# 2) Setup repos, deps, and paths
import os, sys, subprocess


def run(cmd, cwd=None):
    print('+', ' '.join(cmd))
    subprocess.check_call(cmd, cwd=cwd)

if not FYP_REPO.exists():
    run(['git', 'clone', 'https://github.com/spinelessknave8/FYP_code.git', str(FYP_REPO)])
else:
    run(['git', 'pull', '--ff-only'], cwd=str(FYP_REPO))

if not CFLOW_REPO.exists():
    run(['git', 'clone', 'https://github.com/gudovskiy/cflow-ad.git', str(CFLOW_REPO)])
else:
    run(['git', 'fetch', '--all', '--prune'], cwd=str(CFLOW_REPO))
    run(['git', 'checkout', 'master'], cwd=str(CFLOW_REPO))
    run(['git', 'reset', '--hard', 'origin/master'], cwd=str(CFLOW_REPO))

run([sys.executable, '-m', 'pip', 'install', '-q', 'timm==0.9.7', 'FrEIA==0.2', 'scikit-learn==1.3.2'])

os.chdir(str(FYP_REPO))
if str(FYP_REPO / 'severstral-osr' / 'src') not in sys.path:
    sys.path.insert(0, str(FYP_REPO / 'severstral-osr' / 'src'))

print('Setup complete.')


In [ ]:
# 3) Drive + dataset checks
import json

required = [DATA_ROOT / 'train.csv', DATA_ROOT / 'train_images']
for p in required:
    if not p.exists():
        raise FileNotFoundError(f'Missing required dataset path: {p}')

print('Dataset root OK:', DATA_ROOT)


In [ ]:
# 4) Build split manifests (a,b,c,d) with pilot/full sizing
import json, random
from collections import defaultdict
from pathlib import Path
import yaml

from data import collect_single_label_defect_samples, collect_normal_image_paths, stratified_split

rng = random.Random(SEED)
manifest_dir = OUT_ROOT / 'manifests'
manifest_dir.mkdir(parents=True, exist_ok=True)

single = collect_single_label_defect_samples(str(DATA_ROOT), 'train.csv', 'train_images')
normal_paths_all = collect_normal_image_paths(str(DATA_ROOT), 'train.csv', 'train_images')

all_classes = ['Class_1', 'Class_2', 'Class_3', 'Class_4']

if PILOT_MODE:
    cfg_sizes = {
        'normal_train': 240,
        'normal_val': 80,
        'normal_test': 80,
        'known_train_per_class': 80,
        'known_val_per_class': 30,
        'known_test_per_class': 30,
        'unknown_val': 60,
        'unknown_test': 90,
    }
else:
    cfg_sizes = {
        'normal_train': 1200,
        'normal_val': 300,
        'normal_test': 300,
        'known_train_per_class': 300,
        'known_val_per_class': 100,
        'known_test_per_class': 100,
        'unknown_val': 300,
        'unknown_test': 300,
    }


def take_per_class(samples, per_class, seed):
    r = random.Random(seed)
    by = defaultdict(list)
    for p, c in samples:
        by[c].append((p, c))
    out = []
    for c in sorted(by):
        r.shuffle(by[c])
        out.extend(by[c][:per_class])
    return out

split_manifests = {}
for idx, split in enumerate(SPLITS):
    split_yaml = yaml.safe_load((FYP_REPO / 'severstral-osr' / 'configs' / f'split_{split}.yaml').read_text())
    known_classes = split_yaml['known_classes']
    unknown_class = [c for c in all_classes if c not in known_classes][0]

    known_all = [s for s in single if s[1] in known_classes]
    unknown_all = [s for s in single if s[1] == unknown_class]

    ktr_all, kva_all, kte_all = stratified_split(known_all, train_ratio=0.7, val_ratio=0.15, seed=SEED + idx)
    known_train = take_per_class(ktr_all, cfg_sizes['known_train_per_class'], SEED + idx)
    known_val = take_per_class(kva_all, cfg_sizes['known_val_per_class'], SEED + idx)
    known_test = take_per_class(kte_all, cfg_sizes['known_test_per_class'], SEED + idx)

    rr = random.Random(SEED + idx)
    rr.shuffle(unknown_all)
    unknown_val = unknown_all[:cfg_sizes['unknown_val']]
    unknown_test = unknown_all[cfg_sizes['unknown_val']:cfg_sizes['unknown_val'] + cfg_sizes['unknown_test']]

    normals = normal_paths_all[:]
    rr.shuffle(normals)
    ntr = normals[:cfg_sizes['normal_train']]
    nva = normals[cfg_sizes['normal_train']:cfg_sizes['normal_train'] + cfg_sizes['normal_val']]
    nts = normals[cfg_sizes['normal_train'] + cfg_sizes['normal_val']:cfg_sizes['normal_train'] + cfg_sizes['normal_val'] + cfg_sizes['normal_test']]

    manifest = {
        'split_name': f'split_{split}',
        'known_classes': known_classes,
        'unknown_class': unknown_class,
        'normal_train': ntr,
        'normal_val': nva,
        'normal_test': nts,
        'known_train': [{'path': p, 'label': c} for p, c in known_train],
        'known_val': [{'path': p, 'label': c} for p, c in known_val],
        'known_test': [{'path': p, 'label': c} for p, c in known_test],
        'unknown_val': [{'path': p, 'label': c} for p, c in unknown_val],
        'unknown_test': [{'path': p, 'label': c} for p, c in unknown_test],
    }

    outp = manifest_dir / f'split_{split}.json'
    outp.write_text(json.dumps(manifest, indent=2))
    split_manifests[split] = outp
    print(f"split_{split}:", {
        'known_classes': known_classes,
        'normal_train': len(ntr),
        'normal_val': len(nva),
        'normal_test': len(nts),
        'known_train': len(known_train),
        'known_val': len(known_val),
        'known_test': len(known_test),
        'unknown_val': len(unknown_val),
        'unknown_test': len(unknown_test),
    })


In [ ]:
# 5) Patch CFLOW repo safely (reset + compatibility + score export)
import subprocess, sys
from pathlib import Path

# hard reset local edits from prior runs
subprocess.check_call(['git', 'checkout', '--', 'main.py', 'train.py', 'custom_datasets/loader.py'], cwd=str(CFLOW_REPO))

loader_py = CFLOW_REPO / 'custom_datasets' / 'loader.py'
loader_txt = loader_py.read_text()
loader_txt = loader_txt.replace('Image.ANTIALIAS', 'Image.Resampling.LANCZOS')
loader_py.write_text(loader_txt)

train_py = CFLOW_REPO / 'train.py'
train_txt = train_py.read_text()
marker = 'scores_run{c.run_name}_{c.class_name}_epoch{epoch}.npz'
if marker not in train_txt:
    anchor = '        det_roc_auc = roc_auc_score(gt_label, score_label)\n'
    inject = (
        "        det_roc_auc = roc_auc_score(gt_label, score_label)\n"
        "        np.savez(\n"
        "            os.path.join('./results', f'scores_run{c.run_name}_{c.class_name}_epoch{epoch}.npz'),\n"
        "            score_label=score_label.astype(np.float32),\n"
        "            gt_label=gt_label.astype(np.uint8),\n"
        "        )\n"
    )
    if anchor not in train_txt:
        raise RuntimeError('Could not find score anchor in train.py; patch aborted.')
    train_txt = train_txt.replace(anchor, inject, 1)
    train_py.write_text(train_txt)

# syntax guards
subprocess.check_call([sys.executable, '-m', 'py_compile', str(CFLOW_REPO / 'main.py')])
subprocess.check_call([sys.executable, '-m', 'py_compile', str(CFLOW_REPO / 'train.py')])
subprocess.check_call([sys.executable, '-m', 'py_compile', str(CFLOW_REPO / 'custom_datasets' / 'loader.py')])

print('CFLOW patching complete and compiled.')


In [ ]:
# 6) Materialize MVTec-like CFLOW datasets per split (main/val) with caching
import os, json, shutil
from pathlib import Path
from PIL import Image

build_root = OUT_ROOT / 'cflow_datasets'
build_root.mkdir(parents=True, exist_ok=True)


def to_png(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    img = Image.open(src).convert('RGB')
    img.save(dst, format='PNG')


def make_mask_like(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    img = Image.open(src)
    m = Image.new('L', img.size, color=255)
    m.save(dst, format='PNG')


def build_mvtec_layout(dst_root, normal_train, normal_test, known_def, unknown_def):
    if dst_root.exists():
        shutil.rmtree(dst_root)

    # train good
    for i, p in enumerate(normal_train):
        to_png(p, dst_root / 'train' / 'good' / f'{i:06d}.png')

    # test good
    for i, p in enumerate(normal_test):
        to_png(p, dst_root / 'test' / 'good' / f'{i:06d}.png')

    # test known defect
    for i, d in enumerate(known_def):
        src = d['path']
        stem = f'{i:06d}'
        img_path = dst_root / 'test' / 'known_defect' / f'{stem}.png'
        to_png(src, img_path)
        make_mask_like(src, dst_root / 'ground_truth' / 'known_defect' / f'{stem}_mask.png')

    # test unknown defect
    for i, d in enumerate(unknown_def):
        src = d['path']
        stem = f'{i:06d}'
        img_path = dst_root / 'test' / 'unknown_defect' / f'{stem}.png'
        to_png(src, img_path)
        make_mask_like(src, dst_root / 'ground_truth' / 'unknown_defect' / f'{stem}_mask.png')


runtime_manifest_dir = OUT_ROOT / 'runtime_manifests'
runtime_manifest_dir.mkdir(parents=True, exist_ok=True)

for split in SPLITS:
    m = json.loads((manifest_dir / f'split_{split}.json').read_text())
    split_root = build_root / f'split_{split}'
    split_root.mkdir(parents=True, exist_ok=True)

    main_root = split_root / 'main'
    val_root = split_root / 'val'

    build_mvtec_layout(
        main_root,
        normal_train=m['normal_train'],
        normal_test=m['normal_test'],
        known_def=m['known_test'],
        unknown_def=m['unknown_test'],
    )
    build_mvtec_layout(
        val_root,
        normal_train=m['normal_train'],
        normal_test=m['normal_val'],
        known_def=m['known_val'],
        unknown_def=m['unknown_val'],
    )

    rm = {
        'split': f'split_{split}',
        'main_counts': {
            'normal': len(m['normal_test']),
            'known': len(m['known_test']),
            'unknown': len(m['unknown_test']),
        },
        'val_counts': {
            'normal': len(m['normal_val']),
            'known': len(m['known_val']),
            'unknown': len(m['unknown_val']),
        },
        'known_test_labels': [x['label'] for x in m['known_test']],
    }
    (runtime_manifest_dir / f'split_{split}.json').write_text(json.dumps(rm, indent=2))

print('Built MVTec-like datasets at', build_root)


In [ ]:
# 7) Run stage-1 CFLOW for all splits (main + val) with artifact reuse
import os, re, json, time, shutil, subprocess
from pathlib import Path

mvtec_root = CFLOW_REPO / 'data' / 'MVTec-AD'
mvtec_root.mkdir(parents=True, exist_ok=True)
runtime_target = mvtec_root / CFLOW_RUNTIME_CLASS

stage1_dir = OUT_ROOT / 'stage1_scores'
stage1_dir.mkdir(parents=True, exist_ok=True)


def latest_score_npz(run_id):
    cands = sorted((CFLOW_REPO / 'results').glob(f'scores_run{run_id}_{CFLOW_RUNTIME_CLASS}_epoch*.npz'))
    return cands[-1] if cands else None


def swap_in(src_root):
    backup = None
    if runtime_target.exists():
        backup = mvtec_root / f'{CFLOW_RUNTIME_CLASS}_backup'
        if backup.exists():
            shutil.rmtree(backup)
        os.rename(runtime_target, backup)
    os.rename(src_root, runtime_target)
    return backup


def swap_out(src_root, backup):
    if runtime_target.exists():
        os.rename(runtime_target, src_root)
    if backup is not None and backup.exists():
        os.rename(backup, runtime_target)


for sidx, split in enumerate(SPLITS):
    print(f'\n===== Stage1 {split} =====')
    rt = json.loads((runtime_manifest_dir / f'split_{split}.json').read_text())
    split_root = build_root / f'split_{split}'

    run_main = 5000 + sidx
    run_val = 6000 + sidx

    out_main = stage1_dir / f'split_{split}_main.npz'
    out_val = stage1_dir / f'split_{split}_val.npz'

    if out_main.exists() and out_val.exists():
        print('Reuse existing stage1 score files for split', split)
        continue

    # main run
    src_main = split_root / 'main'
    bkp = swap_in(src_main)
    try:
        if latest_score_npz(run_main) is None:
            subprocess.check_call([
                'python', 'main.py', '--dataset', 'mvtec', '--class-name', CFLOW_RUNTIME_CLASS,
                '--action-type', 'norm-train', '--run-name', str(run_main), '--batch-size', '8',
                '--meta-epochs', '1' if PILOT_MODE else '8', '--sub-epochs', '1' if PILOT_MODE else '4',
                '--workers', '2', '--input-size', '256', '--gpu', '0'
            ], cwd=str(CFLOW_REPO))
        npz_main = latest_score_npz(run_main)
        if npz_main is None:
            raise RuntimeError(f'Missing main score artifact for split {split}')
        z = dict(**__import__('numpy').load(npz_main))
        n = rt['main_counts']['normal']; k = rt['main_counts']['known']; u = rt['main_counts']['unknown']
        score = z['score_label'].reshape(-1)
        if len(score) != (n + k + u):
            raise RuntimeError(f'Unexpected main score length for split {split}: {len(score)} vs {n+k+u}')
        __import__('numpy').savez(out_main,
            normal_scores=score[:n].astype('float32'),
            known_scores=score[n:n+k].astype('float32'),
            unknown_scores=score[n+k:n+k+u].astype('float32')
        )
    finally:
        swap_out(src_main, bkp)

    # val run
    src_val = split_root / 'val'
    bkp = swap_in(src_val)
    try:
        if latest_score_npz(run_val) is None:
            subprocess.check_call([
                'python', 'main.py', '--dataset', 'mvtec', '--class-name', CFLOW_RUNTIME_CLASS,
                '--action-type', 'norm-train', '--run-name', str(run_val), '--batch-size', '8',
                '--meta-epochs', '1', '--sub-epochs', '1', '--workers', '2', '--input-size', '256', '--gpu', '0'
            ], cwd=str(CFLOW_REPO))
        npz_val = latest_score_npz(run_val)
        if npz_val is None:
            raise RuntimeError(f'Missing val score artifact for split {split}')
        z = dict(**__import__('numpy').load(npz_val))
        n = rt['val_counts']['normal']; k = rt['val_counts']['known']; u = rt['val_counts']['unknown']
        score = z['score_label'].reshape(-1)
        if len(score) != (n + k + u):
            raise RuntimeError(f'Unexpected val score length for split {split}: {len(score)} vs {n+k+u}')
        __import__('numpy').savez(out_val,
            normal_scores=score[:n].astype('float32'),
            known_scores=score[n:n+k].astype('float32'),
            unknown_scores=score[n+k:n+k+u].astype('float32')
        )
    finally:
        swap_out(src_val, bkp)

    print('Saved stage1:', out_main.name, out_val.name)


In [ ]:
# 8) Train/reuse Stage-2 ResNet50 per split (known-class classifier)
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from pathlib import Path
from PIL import Image

from data import ImageLabelDataset
from src.models.resnet50 import build_resnet50

device = 'cuda' if torch.cuda.is_available() else 'cpu'

TF = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

stage2_dir = OUT_ROOT / 'stage2'
stage2_dir.mkdir(parents=True, exist_ok=True)


def infer_logits(model, items, class_to_idx):
    ds = ImageLabelDataset([(x['path'], x['label']) for x in items], class_to_idx=class_to_idx, transform=TF)
    dl = DataLoader(ds, batch_size=32, shuffle=False, num_workers=2)
    logits = []
    labels = []
    model.eval()
    with torch.no_grad():
        for x, y in dl:
            x = x.to(device)
            out = model(x).cpu().numpy()
            logits.append(out)
            labels.append(y.numpy())
    return np.concatenate(logits, axis=0), np.concatenate(labels, axis=0)

for split in SPLITS:
    m = json.loads((manifest_dir / f'split_{split}.json').read_text())
    model_path = stage2_dir / f'split_{split}_resnet50.pt'
    cache_path = stage2_dir / f'split_{split}_logits.npz'

    known_classes = m['known_classes']
    class_to_idx = {c: i for i, c in enumerate(known_classes)}

    if not model_path.exists():
        train_ds = ImageLabelDataset([(x['path'], x['label']) for x in m['known_train']], class_to_idx=class_to_idx, transform=TF)
        val_ds = ImageLabelDataset([(x['path'], x['label']) for x in m['known_val']], class_to_idx=class_to_idx, transform=TF)
        tr = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
        va = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

        model = build_resnet50(num_classes=len(class_to_idx), pretrained=True).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
        crit = nn.CrossEntropyLoss()

        epochs = 2 if PILOT_MODE else 8
        best_acc = -1.0
        best_state = None
        for ep in range(epochs):
            model.train()
            for x, y in tr:
                x, y = x.to(device), y.to(device)
                opt.zero_grad()
                loss = crit(model(x), y)
                loss.backward()
                opt.step()

            model.eval()
            c = 0; n = 0
            with torch.no_grad():
                for x, y in va:
                    x, y = x.to(device), y.to(device)
                    pred = model(x).argmax(1)
                    c += (pred == y).sum().item(); n += y.size(0)
            acc = c / max(1, n)
            if acc > best_acc:
                best_acc = acc
                best_state = {k: v.cpu() for k, v in model.state_dict().items()}

        model.load_state_dict(best_state)
        torch.save({'state_dict': model.state_dict(), 'known_classes': known_classes}, model_path)
        print(f'split_{split}: stage2 model saved, best_val_acc={best_acc:.4f}')
    else:
        print(f'split_{split}: reusing stage2 model')

    if cache_path.exists():
        print(f'split_{split}: reusing stage2 logits cache')
        continue

    ck = torch.load(model_path, map_location='cpu')
    model = build_resnet50(num_classes=len(known_classes), pretrained=False)
    model.load_state_dict(ck['state_dict'])
    model = model.to(device).eval()

    log_kval, y_kval = infer_logits(model, m['known_val'], class_to_idx)
    log_ktest, y_ktest = infer_logits(model, m['known_test'], class_to_idx)

    # unknown has labels not in class_to_idx; run with dummy idx map
    unk_items = [(x['path'], known_classes[0]) for x in m['unknown_test']]
    from data import ImageLabelDataset as ILD
    unk_ds = ILD(unk_items, class_to_idx={known_classes[0]: 0}, transform=TF)
    unk_dl = DataLoader(unk_ds, batch_size=32, shuffle=False, num_workers=2)
    log_utest = []
    with torch.no_grad():
        for x, _ in unk_dl:
            x = x.to(device)
            log_utest.append(model(x).cpu().numpy())
    log_utest = np.concatenate(log_utest, axis=0)

    np.savez(cache_path,
        logits_known_val=log_kval.astype('float32'),
        labels_known_val=y_kval.astype('int64'),
        logits_known_test=log_ktest.astype('float32'),
        labels_known_test=y_ktest.astype('int64'),
        logits_unknown_test=log_utest.astype('float32')
    )
    print(f'split_{split}: stage2 logits cached')


In [ ]:
# 9) Final protocol evaluation (per split x FPR), save CSVs, display tables
import json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

rows = []

for split in SPLITS:
    m = json.loads((manifest_dir / f'split_{split}.json').read_text())
    rt = json.loads((runtime_manifest_dir / f'split_{split}.json').read_text())

    st1_main = np.load(stage1_dir / f'split_{split}_main.npz')
    st1_val = np.load(stage1_dir / f'split_{split}_val.npz')

    s_n_test = st1_main['normal_scores']
    s_k_test = st1_main['known_scores']
    s_u_test = st1_main['unknown_scores']

    # strict validation-only tau calibration (normal-val only)
    s_n_val = st1_val['normal_scores']

    st2 = np.load(stage2_dir / f'split_{split}_logits.npz')
    lk_val = st2['logits_known_val']
    lk_test = st2['logits_known_test']
    lu_test = st2['logits_unknown_test']
    yk_test = st2['labels_known_test']

    def softmax(x):
        x = x - x.max(axis=1, keepdims=True)
        ex = np.exp(x)
        return ex / ex.sum(axis=1, keepdims=True)

    pk_val = softmax(lk_val)
    pk_test = softmax(lk_test)
    pu_test = softmax(lu_test)

    conf_k_val = pk_val.max(axis=1)
    conf_k_test = pk_test.max(axis=1)
    conf_u_test = pu_test.max(axis=1)

    pred_k_test = pk_test.argmax(axis=1)

    kappa = float(np.quantile(conf_k_val, STAGE2_UNKNOWN_Q))

    y_true = np.concatenate([
        np.zeros_like(s_n_test, dtype=np.int64),
        np.ones_like(s_k_test, dtype=np.int64),
        np.ones_like(s_u_test, dtype=np.int64),
    ])
    y_score = np.concatenate([s_n_test, s_k_test, s_u_test])
    auroc = float(roc_auc_score(y_true, y_score))

    for fpr in FPR_GRID:
        tau = float(np.quantile(s_n_val, 1.0 - fpr))

        n_def = s_n_test > tau
        k_def = s_k_test > tau
        u_def = s_u_test > tau

        # two-stage decisions
        k_unknown = conf_k_test < kappa
        u_unknown = conf_u_test < kappa

        # known success: stage1 flags defect AND stage2 assigns correct known class (not unknown)
        k_success = k_def & (~k_unknown) & (pred_k_test == yk_test)

        # unknown success: stage1 flags defect AND stage2 rejects as unknown
        u_success = u_def & u_unknown

        row = {
            'split': f'split_{split}',
            'fpr_target': float(fpr),
            'AUROC_defect_screening': auroc,
            'TPR_defect@FPR': float(np.mean(np.concatenate([k_def, u_def]))),
            'TPR_unknown@FPR': float(np.mean(u_def)),
            'FPR_known_as_def@FPR': float(np.mean(k_def)),
            'FPR_normal_realized': float(np.mean(n_def)),
            'two_stage_known_success': float(np.mean(k_success)),
            'two_stage_unknown_success': float(np.mean(u_success)),
            'stage2_unknown_rate_known_all': float(np.mean(k_unknown)),
            'stage2_unknown_rate_unknown_all': float(np.mean(u_unknown)),
            'tau_stage1': tau,
            'kappa_stage2': kappa,
        }
        rows.append(row)

summary_df = pd.DataFrame(rows).sort_values(['split', 'fpr_target']).reset_index(drop=True)

metric_cols = [
    'AUROC_defect_screening', 'TPR_defect@FPR', 'TPR_unknown@FPR', 'FPR_known_as_def@FPR',
    'FPR_normal_realized', 'two_stage_known_success', 'two_stage_unknown_success',
    'stage2_unknown_rate_known_all', 'stage2_unknown_rate_unknown_all'
]

mean_std = summary_df.groupby('fpr_target', as_index=False)[metric_cols].agg(['mean', 'std'])
mean_std.columns = ['fpr_target'] + [f'{m}_{s}' for m, s in mean_std.columns.tolist()[1:]]

summary_csv = OUT_ROOT / 'cflow_two_stage_summary.csv'
mean_std_csv = OUT_ROOT / 'cflow_two_stage_mean_std.csv'
summary_df.to_csv(summary_csv, index=False)
mean_std.to_csv(mean_std_csv, index=False)

print('Per-split table:')
display(summary_df)
print('Mean/std table:')
display(mean_std)

print('\nSaved CSVs:')
print(summary_csv)
print(mean_std_csv)
print('Summary rows:', len(summary_df))
print('Mean/std rows:', len(mean_std))
print('FPR grid includes 0.15:', 0.15 in sorted(summary_df['fpr_target'].unique().tolist()))


In [ ]:
# 10) Optional plots
import pandas as pd
import matplotlib.pyplot as plt

summary = pd.read_csv(OUT_ROOT / 'cflow_two_stage_summary.csv')
mean_std = pd.read_csv(OUT_ROOT / 'cflow_two_stage_mean_std.csv')

plt.figure(figsize=(7,4))
for split, g in summary.groupby('split'):
    g = g.sort_values('fpr_target')
    plt.plot(g['fpr_target'], g['TPR_defect@FPR'], marker='o', alpha=0.5, label=split)
plt.xlabel('FPR target')
plt.ylabel('TPR_defect@FPR')
plt.title('CFLOW two-stage per split')
plt.grid(alpha=0.3)
plt.legend()
plot1 = OUT_ROOT / 'plot_cflow_two_stage_tpr_defect_per_split.png'
plt.tight_layout(); plt.savefig(plot1, dpi=160); plt.show()

plt.figure(figsize=(7,4))
x = mean_std['fpr_target']
y = mean_std['TPR_defect@FPR_mean']
e = mean_std['TPR_defect@FPR_std'].fillna(0)
plt.errorbar(x, y, yerr=e, marker='o', capsize=3)
plt.xlabel('FPR target')
plt.ylabel('TPR_defect@FPR mean±std')
plt.title('CFLOW two-stage mean/std across splits')
plt.grid(alpha=0.3)
plot2 = OUT_ROOT / 'plot_cflow_two_stage_tpr_defect_mean_std.png'
plt.tight_layout(); plt.savefig(plot2, dpi=160); plt.show()

print('Saved plots:')
print(plot1)
print(plot2)


## Run Order
1. Run cells `0` to `10` top-to-bottom.
2. Keep `PILOT_MODE=True` first to validate end-to-end quickly.
3. Set `PILOT_MODE=False` and rerun from cell `4` onward for heavier run.
4. Final required files are:
   - `/content/drive/MyDrive/fyp_outputs/severstral_cflow_two_stage_full/cflow_two_stage_summary.csv`
   - `/content/drive/MyDrive/fyp_outputs/severstral_cflow_two_stage_full/cflow_two_stage_mean_std.csv`